In [1]:
import sys
sys.path.append("..")
from src.data import load_dataset, split_data

for name in ["telco", "bank"]:
    df = load_dataset(name)
    X_train, X_test, y_train, y_test = split_data(df)
    print(name, "train:", X_train.shape, "test:", X_test.shape)
    print("  churn rate train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))

telco train: (5634, 19) test: (1409, 19)
  churn rate train: 0.265  test: 0.265
bank train: (8000, 10) test: (2000, 10)
  churn rate train: 0.204  test: 0.204


In [2]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from src.models import make_logreg

scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall",
           "f1": "f1", "roc_auc": "roc_auc", "pr_auc": "average_precision"}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name in ["telco", "bank"]:
    df = load_dataset(name)
    X_train, X_test, y_train, y_test = split_data(df)
    res = cross_validate(make_logreg(X_train), X_train, y_train, cv=cv, scoring=scoring)
    print(name, {k[5:]: round(v.mean(), 3) for k, v in res.items() if k.startswith("test_")})
    print(name, "accuracy of always guessing 'stays':", round(1 - y_train.mean(), 3))

telco {'accuracy': np.float64(0.802), 'precision': np.float64(0.653), 'recall': np.float64(0.544), 'f1': np.float64(0.593), 'roc_auc': np.float64(0.846), 'pr_auc': np.float64(0.661)}
telco accuracy of always guessing 'stays': 0.735
bank {'accuracy': np.float64(0.81), 'precision': np.float64(0.597), 'recall': np.float64(0.214), 'f1': np.float64(0.315), 'roc_auc': np.float64(0.763), 'pr_auc': np.float64(0.469)}
bank accuracy of always guessing 'stays': 0.796


In [ ]:
import pandas as pd
from src.data import load_dataset, split_data
from src.models import make_logreg, make_rf, make_xgb
from src.evaluate import cv_report

results = []
for name in ["telco", "bank"]:
    X_train, X_test, y_train, y_test = split_data(load_dataset(name))
    for model_name, maker in [("LogReg", make_logreg), ("RandomForest", make_rf), ("XGBoost", make_xgb)]:
        scores = cv_report(maker(X_train), X_train, y_train)
        results.append({"dataset": name, "model": model_name, **scores})

pd.DataFrame(results)
pd.DataFrame(results).to_csv("../results/model_comparison_baseline.csv", index=False)

,dataset,model,precision,recall,f1,roc_auc,pr_auc
0,telco,LogReg,0.653,0.544,0.593,0.846,0.661
1,telco,RandomForest,0.668,0.496,0.569,0.845,0.659
2,telco,XGBoost,0.659,0.524,0.584,0.844,0.666
3,bank,LogReg,0.597,0.214,0.315,0.763,0.469
4,bank,RandomForest,0.785,0.446,0.569,0.859,0.684
5,bank,XGBoost,0.753,0.478,0.585,0.865,0.701
